📊 01. Análisis Exploratorio de Datos (EDA) - Segmentación de Clientes
Este documento contiene la estructura formal del notebook de análisis exploratorio para el proyecto de clasificación multiclase de clientes del sector automovilístico.

1. Carga de Datos e Infraestructura 📦
Descripción:

En este apartado se realiza la configuración del entorno gráfico (seaborn y matplotlib) y la importación de los datos directamente desde Kaggle mediante la librería kagglehub. Los archivos Train.csv y Test.csv se copian automáticamente a la carpeta local data/ para garantizar la reproducibilidad y evitar descargas innecesarias en cada ejecución.

In [1]:
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

# Configuración del estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Rutas locales dentro de la carpeta del proyecto
ruta_train_local = "data/Train.csv"
ruta_test_local = "data/Test.csv"

# Comprobar si los archivos ya existen en la carpeta local 'data/'
if not (os.path.exists(ruta_train_local) and os.path.exists(ruta_test_local)):
    print("Descargando dataset desde Kaggle...")
    path = kagglehub.dataset_download("kaushiksuresh147/customer-segmentation")
    os.makedirs('data', exist_ok=True)
    
    # Copiar Train.csv y Test.csv a la carpeta local 'data'
    shutil.copy(os.path.join(path, "Train.csv"), ruta_train_local)
    shutil.copy(os.path.join(path, "Test.csv"), ruta_test_local)
    print("¡Descarga y copia completadas en la carpeta data/!")
else:
    print("Los archivos locales Train.csv y Test.csv ya existen en data/.")

# Cargar los datasets
train_df = pd.read_csv(ruta_train_local)
test_df = pd.read_csv(ruta_test_local)

print(f"Dataset Train listo: {train_df.shape[0]} filas y {train_df.shape[1]} columnas.")
print(f"Dataset Test listo: {test_df.shape[0]} filas y {test_df.shape[1]} columnas.")

train_df.head()

Descargando dataset desde Kaggle...


100%|██████████| 99.9k/99.9k [00:00<00:00, 503kB/s]

Extracting files...


¡Descarga y copia completadas en la carpeta data/!
Dataset Train listo: 8068 filas y 11 columnas.
Dataset Test listo: 2627 filas y 11 columnas.


,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


2. Inspección General y Diagnóstico de Nulos 🔎
Descripción:

Se efectúa el primer diagnóstico de las estructuras de los datasets de entrenamiento (8,068 registros) y test (2,627 registros). Se identifican los tipos de datos de cada columna y se calcula el volumen y porcentaje de valores nulos, destacando variables críticas como Work_Experience (10.28% nulos) y Family_Size (4.15% nulos).

In [2]:
# Resumen de información general y nulos
info_df = pd.DataFrame({
    'Tipo': train_df.dtypes,
    'Valores Nulos': train_df.isnull().sum(),
    '% Nulos': (train_df.isnull().sum() / len(train_df) * 100).round(2),
    'Valores Únicos': train_df.nunique()
})

print("--- RESUMEN DE VARIABLES EN TRAIN ---")
info_df

--- RESUMEN DE VARIABLES EN TRAIN ---


,Tipo,Valores Nulos,% Nulos,Valores Únicos
ID,int64,0,0.00,8068
Gender,str,0,0.00,2
Ever_Married,str,140,1.74,2
Age,int64,0,0.00,67
Graduated,str,78,0.97,2
Profession,str,124,1.54,9
Work_Experience,float64,829,10.28,15
Spending_Score,str,0,0.00,3
Family_Size,float64,335,4.15,9
Var_1,str,76,0.94,7


3. Análisis Univariado (Distribución de Variables) 📈Descripción:Estudio individual de la variable objetivo (Segmentation) y del resto de características demográficas. Se comprueba que la variable a predecir presenta un buen equilibrio entre las cuatro categorías ($D: 28.11\%$, $A: 24.44\%$, $C: 24.42\%$, $B: 23.03\%$), lo que simplifica las métricas de evaluación del modelo.

4. Análisis Bivariado y Profiling de Segmentos 👥
Descripción:

Exploración de las interacciones entre los segmentos de clientes y sus atributos clave (Age, Spending_Score, Profession, Ever_Married). Permite caracterizar los perfiles principales del negocio:

Segmento D: Clientes más jóvenes (mediana de 29 años), con nivel de gasto bajo (Low) y fuerte concentración en el sector Healthcare.

Segmentos B y C: Clientes de mayor edad (medianas entre 46 y 49 años), con mayor representación de profesionales independientes (Artist) y niveles de gasto medio/alto (Average/High).

5. Estrategia de Imputación y Preprocesamiento 🛠️
Descripción:

Definición formal del plan de tratamiento de nulos (imputación por modas y medianas agrupadas por profesión o estado civil) y estrategias de codificación (encoding) para variables categóricas, preparando las transformaciones necesarias para la fase de modelado de Machine Learning.

6. Exportación del Dataset Limpio 💾
Descripción:

Generación de los archivos procesados train_clean.csv y test_clean.csv en el directorio data/. Este entregable compartido servirá como base estandarizada para que todo el equipo pueda entrenar sus modelos de clasificación sin inconsistencias en los datos.